In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [5]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [6]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [7]:
def get_item_idx(terms):
    assert type(terms) == list
    
    a_name = find_match_using_terms(terms, cat2idx)
    a = cat2idx[a_name]
    
    print("Artist:", a_name)
    print("*" * 20)
    
    assert artist_mat[:, a].sum() > 0
    
    return a

In [8]:
mat = artist_mat
mat = csr_array(mat)

In [9]:
mat.shape

(1000000, 295860)

In [10]:
X = mat.T @ mat

In [11]:
X.shape

(295860, 295860)

In [12]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize_scalar

def nan_to_const(x, const):
    return np.where(~np.isfinite(x), const, x)

class SppmiOptimizerCache:
    def __init__(self, X):
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()

def optimize_alpha_nested_groups(cache, item_groups, ranking_type, mode="direct", 
                                 symmetrize=False, gamma=1.0, tau=2.0, 
                                 zero_diag=True, bounds=(1e-4, 2.0), return_scores=False):
    """
    Optimizes alpha across multiple independent groups of items.
    Calculates the average pairwise ranking for each group separately, 
    then minimizes the global average of those group metrics.
    """
    # 1. Enforce validation constraints
    for i, group in enumerate(item_groups):
        assert len(group) >= 2, f"Group at index {i} has fewer than 2 items: {group}. All groups must contain at least 2 items."

    X = cache.X_csc
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums_raw = cache.row_sums_raw
    col_sums_raw = cache.col_sums_raw
    
    # Deduplicate all items across all groups to maximize cache efficiency
    unique_items = list(set(idx for group in item_groups for idx in group))

    # 2. Precompute structural vectors only once per unique item
    item_profiles = {}
    for idx in unique_items:
        col = X[:, idx:idx+1]
        profile = {
            'indices': col.indices,
            'data': col.data.astype(np.float32)
        }
        if mode == "dot_product":
            profile['raw_dots'] = (X[:, idx].T @ X).toarray().flatten()
        item_profiles[idx] = profile

    def get_ranking_score(idx, alpha):
        N_smoothed = N_raw + (alpha * rows * cols)
        prof = item_profiles[idx]
        curr_rows, curr_data = prof['indices'], prof['data']
        raw_counts = np.zeros(rows, dtype=np.float32)
        
        if mode == "direct":
            P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            
            P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
            
            lift = P_xy / (P_x * P_y_idx)
            px_given_y = P_xy / P_y_idx  
            py_given_x = P_xy / P_x      
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            raw_counts[curr_rows] = curr_data
        else:
            P_x = (col_sums_raw + (alpha * rows)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            raw_dots = prof['raw_dots']
            num_counts = raw_dots + (alpha * col_sums_raw[idx]) + (alpha * col_sums_raw) + (rows * (alpha ** 2))
            
            lift = num_counts / ((col_sums_raw[idx] + (alpha * rows)) * (col_sums_raw + (alpha * rows)))
            px_given_y = num_counts / (N_smoothed * (col_sums_raw[idx] + (alpha * rows)))
            py_given_x = num_counts / (N_smoothed * (col_sums_raw + (alpha * rows)))
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            raw_counts = raw_dots

        if ranking_type == "soft_power_lift" and mode == "direct":
            soft_power_lift = P_xy / ((P_x ** gamma) * (P_y_idx ** gamma))
        else:
            soft_power_lift = lift

        if ranking_type == "weighted_lift":
            forward = lift * px_given_y
            backward = lift * py_given_x
            score = np.sqrt(forward * backward) if symmetrize else forward
        elif ranking_type == "normalized_lift":
            tuned_normalizer = np.minimum(1.0 / (P_x ** gamma), 1.0 / (P_y_idx ** gamma))
            score = lift / tuned_normalizer
        elif ranking_type == "normalized_weighted_lift":
            forward = (lift / lift_normalizer) * px_given_y
            backward = (lift / lift_normalizer) * py_given_x
            score = np.sqrt(forward * backward) if symmetrize else forward
        elif ranking_type == "soft_power_lift":
            score = soft_power_lift
        elif ranking_type == "npmi_alt":
            score = np.log2(lift) / np.log2(lift_normalizer)
        else:
            score = lift

        if tau > 0:
            score *= (raw_counts / (raw_counts + tau))
            
        if zero_diag:
            score[idx] = 0
            
        return score

    def objective(alpha):
        # Generate raw 1D scoring fields for all unique items present
        current_scores = {idx: get_ranking_score(idx, alpha) for idx in unique_items}
        
        group_scores = []
        
        # Process every cluster/cohort independently
        for group in item_groups:
            total_pairwise_rank = 0.0
            k = len(group)
            
            for u in group:
                scores_u = current_scores[u]
                sort_indices = np.argsort(-scores_u)
                
                # Invert the sorted indices map in O(N) for fast lookup 
                ranks = np.empty_like(sort_indices)
                ranks[sort_indices] = np.arange(len(sort_indices))
                
                for v in group:
                    if u == v:
                        continue
                    total_pairwise_rank += ranks[v]
            
            # Normalize by total evaluation directions in this specific cluster
            group_scores.append(total_pairwise_rank / (k * (k - 1)))
            
        # Return the flat arithmetic average across all groups
        return np.mean(group_scores)

    # Execute optimization via bounded scalar search
    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    
    if return_scores:
        optimized_alpha = result.x
        final_score_dict = {idx: get_ranking_score(idx, optimized_alpha) for idx in unique_items}
        return result, final_score_dict

    return result

In [13]:
def compute_item_scores(cache, a, alpha, ranking_type, mode="direct", 
                        symmetrize=False, gamma=1.0, tau=2.0, zero_diag=True):
    """
    Generates a full 1D scoring profile for a single item 'a' against all items,
    using pre-computed/fitted hyperparameters.
    """
    X = cache.X_csc
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums_raw = cache.row_sums_raw
    col_sums_raw = cache.col_sums_raw

    # Extract sparse details for target item 'a'
    col_a = X[:, a:a+1]
    rows_a, data_a = col_a.indices, col_a.data.astype(np.float32)

    if mode == "dot_product":
        raw_dots_a = (X[:, a].T @ X).toarray().flatten()

    N_smoothed = N_raw + (alpha * rows * cols)
    raw_counts = np.zeros(rows, dtype=np.float32)
    
    if mode == "direct":
        P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
        P_y_idx = (col_sums_raw[a] + (alpha * rows)) / N_smoothed
        
        P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
        P_xy[rows_a] = (data_a + alpha) / N_smoothed
        
        lift = P_xy / (P_x * P_y_idx)
        px_given_y = P_xy / P_y_idx  
        py_given_x = P_xy / P_x      
        lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
        raw_counts[rows_a] = data_a
    else:
        P_x = (col_sums_raw + (alpha * rows)) / N_smoothed
        P_y_idx = (col_sums_raw[a] + (alpha * rows)) / N_smoothed
        num_counts = raw_dots_a + (alpha * col_sums_raw[a]) + (alpha * col_sums_raw) + (rows * (alpha ** 2))
        
        lift = num_counts / ((col_sums_raw[a] + (alpha * rows)) * (col_sums_raw + (alpha * rows)))
        px_given_y = num_counts / (N_smoothed * (col_sums_raw[a] + (alpha * rows)))
        py_given_x = num_counts / (N_smoothed * (col_sums_raw + (alpha * rows)))
        lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
        raw_counts = raw_dots_a

    if ranking_type == "soft_power_lift" and mode == "direct":
        soft_power_lift = P_xy / ((P_x ** gamma) * (P_y_idx ** gamma))
    else:
        soft_power_lift = lift

    if ranking_type == "weighted_lift":
        forward = lift * px_given_y
        backward = lift * py_given_x
        score = np.sqrt(forward * backward) if symmetrize else forward
    elif ranking_type == "normalized_lift":
        tuned_normalizer = np.minimum(1.0 / (P_x ** gamma), 1.0 / (P_y_idx ** gamma))
        score = lift / tuned_normalizer
    elif ranking_type == "normalized_weighted_lift":
        forward = (lift / lift_normalizer) * px_given_y
        backward = (lift / lift_normalizer) * py_given_x
        score = np.sqrt(forward * backward) if symmetrize else forward
    elif ranking_type == "soft_power_lift":
        score = soft_power_lift
    elif ranking_type == "npmi_alt":
        score = np.log2(lift) / np.log2(lift_normalizer)
    else:
        score = lift

    if tau > 0:
        score *= (raw_counts / (raw_counts + tau))
        
    if zero_diag:
        score[a] = 0
        
    return score

In [14]:
# terms = ["Megadeth"]
# terms = ["Havok (spotify:artist:2jw4wgixxa20jls9N3Bdpq)"]
# terms = ["Evile (spotify:artist:1dwrMJAKBiLlj0O4R791Xo)"]
# terms = ["Sonata Arctica"]
# terms = ["Muse (spotify:artist:12Chz98pHFMPJEknJQMWvI)"]
# terms = ["Dream Theater (spotify:artist:2aaLAng2L2aWD2FClzwiep)"]
# terms = ["Saints Go Machine"]
# terms = ["AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)"]
# terms = ["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"]
# terms = ["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"]
# terms = ["Fleet Foxes"]
# terms = ["Vektor (spotify:artist:09mNj9XgCqgg6usfeXOoBg)"]
# terms = ["Dr. Living Dead (spotify:artist:0gLz6azFpZgHyJkJd5yuiM)"]
# terms = ["Lich King (spotify:artist:4rlxS0LeVnHz6z1zp2iJbz)"]
# terms = ["Skeletonwitch (spotify:artist:213mmq3zkNWx7CtfzftTC5)"]
# terms = ["Wintersun (spotify:artist:6ui6SwChan7c1KYBQCqGKV)"]
# terms = ["Chris Poland"]
# terms = ["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]
# terms = ["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]
# terms = ["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]

# a = get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"])
# b = get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"])

In [15]:
# 1. Initialize the global shared data structure
cache = SppmiOptimizerCache(X)

In [16]:
# items = [
#     get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]),
#     get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]),
# #     get_item_idx(["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]),
# ]

In [17]:
item_group_a = [
    get_item_idx(["Highasakite"]),
    get_item_idx(["Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)"]),
]

Artist: Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
********************
Artist: Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
********************


In [18]:
item_group_b = [
    get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]),
    get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]),
#     get_item_idx(["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]),
]

Artist: Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)
********************
Artist: Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)
********************


In [19]:
item_group_c = [
    get_item_idx(["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]),
    get_item_idx(["Megadeth"]),
]

Artist: Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)
********************
Artist: Megadeth (spotify:artist:1Yox196W7bzVNZI7RBaPnf)
********************


In [65]:
nested_items = [
    item_group_a,
    item_group_b,
#     item_group_c,
]

In [66]:
ranking_type = "soft_power_lift"
gamma = 1.4
tau = 4.4

# 2. STEP 1: Fit parameters using your validation pairs (a=42, b=107)
fit_result = optimize_alpha_nested_groups(
    cache,
    nested_items,
    ranking_type=ranking_type, 
    gamma=gamma, 
    tau=tau,
    return_scores=False
)

# Extract your optimal fitted alpha configuration
best_alpha = fit_result.x
print(f"--- Fit Complete ---")
print(f"Target Optimized Alpha: {best_alpha:.6f}\n")
print(fit_result.fun)

--- Fit Complete ---
Target Optimized Alpha: 0.117320

3.75


In [ ]:
a, b = item_group_a

scores_a = compute_item_scores(
    cache, 
    a=a, 
    alpha=best_alpha, 
    ranking_type=ranking_type,
    gamma=gamma, 
    tau=tau,
)

scores_b = compute_item_scores(
    cache, 
    a=b, 
    alpha=best_alpha, 
    ranking_type=ranking_type,
    gamma=gamma, 
    tau=tau,
)

(np.argsort(-scores_a).tolist().index(b) + np.argsort(-scores_b).tolist().index(a))/2

In [124]:
k = 15

# Asgeir
# terms = ["Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)"]
terms = ["Highasakite"]
# terms = ["Elsa & Emilie"]
# terms = ["Susanne Sundfør"]
# terms = ["Billy Joel (spotify:artist:6zFYqv1mOsgBRQbae3JJ9e)"]
# terms = ["Megadeth"]
# terms = ["Havok (spotify:artist:2jw4wgixxa20jls9N3Bdpq)"]
# terms = ["Evile (spotify:artist:1dwrMJAKBiLlj0O4R791Xo)"]
# terms = ["Sonata Arctica"]
# terms = ["Muse (spotify:artist:12Chz98pHFMPJEknJQMWvI)"]
# terms = ["Dream Theater (spotify:artist:2aaLAng2L2aWD2FClzwiep)"]
# terms = ["Saints Go Machine"]
# terms = ["AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)"]
# terms = ["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"]
# terms = ["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"]
# terms = ["Fleet Foxes"]
# terms = ["Vektor (spotify:artist:09mNj9XgCqgg6usfeXOoBg)"]
# terms = ["Dr. Living Dead (spotify:artist:0gLz6azFpZgHyJkJd5yuiM)"]
# terms = ["Lich King (spotify:artist:4rlxS0LeVnHz6z1zp2iJbz)"]
# terms = ["Skeletonwitch (spotify:artist:213mmq3zkNWx7CtfzftTC5)"]
# terms = ["Wintersun (spotify:artist:6ui6SwChan7c1KYBQCqGKV)"]
# terms = ["Chris Poland"]
# terms = ["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]
# terms = ["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]
# terms = ["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]

item_idx = get_item_idx(terms)

scores = compute_item_scores(
    cache, 
    a=item_idx, 
    alpha=best_alpha, 
    ranking_type=ranking_type,
    gamma=gamma, 
    tau=tau,
)

for i in np.argsort(-scores)[:k]:
    print(idx2cat[i])

Artist: Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
********************
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)
Alice Boman (spotify:artist:3WiytRnvoL0kT3oAGl9TCt)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Amason (spotify:artist:4cJKxS7uOPhwb5UQ70sYpN)
Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)
Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)
SOAK (spotify:artist:4PLsMEk2DCRVlVL2a9aZAv)
Phoria (spotify:artist:0HDxlFsXwyrpufs4YgTNMm)
Oh Pep! (spotify:artist:3L9rqEIsNSaOcx2QIstn7v)
